# SECTION 1. SETUP AND INSTALLATIONS


In [2]:
#IMPORTS
import pandas as pd
import numpy as np
import os
import re
import nltk
nltk.download('stopwords')

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# SECTION 2. DATA LOADING AND EXPLORATION

In [4]:
import kagglehub

path = kagglehub.dataset_download("gowrishankarp/newspaper-text-summarization-cnn-dailymail")
inner_path = os.path.join(path, "cnn_dailymail")
print(os.listdir(inner_path))

df = pd.read_csv(os.path.join(inner_path, "train.csv"))
df = df[['article', 'highlights']].dropna()
df = df.sample(5000, random_state=42).reset_index(drop=True)

print(f"Shape: {df.shape}")
df.head(3)

100%|██████████| 503M/503M [04:10<00:00, 2.10MB/s] 

Extracting files...


['test.csv', 'train.csv', 'validation.csv']
Shape: (5000, 2)


,article,highlights
0,By . Mia De Graaf . Britons flocked to beaches...,People enjoyed temperatures of 17C at Brighton...
1,A couple who weighed a combined 32st were sham...,Couple started piling on pounds after the birt...
2,Video footage shows the heart stopping moment ...,A 17-year-old boy suffering lacerations to his...


### EDA - DATA EXPLORATION

In [5]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample Article:\n", df['article'][0][:300])
print("\nSample Highlight:\n", df['highlights'][0])

df['article_len'] = df['article'].apply(lambda x: len(x.split()))
df['highlight_len'] = df['highlights'].apply(lambda x: len(x.split()))

print("\nArticle Length Statistics:")
print(df['article_len'].describe())
print("\nHighlight Length Statistics:")
print(df['highlight_len'].describe())

Shape: (5000, 2)

Columns: ['article', 'highlights']

Sample Article:
 By . Mia De Graaf . Britons flocked to beaches across the southern coast yesterday as millions look set to bask in glorious sunshine today. Temperatures soared to 17C in Brighton and Dorset, with people starting their long weekend in deck chairs by the sea. Figures from Asda suggest the unexpected s

Sample Highlight:
 People enjoyed temperatures of 17C at Brighton beach in West Sussex and Weymouth in Dorset .
Asda claims it will sell a million sausages over long weekend despite night temperatures dropping to minus 1C .
But the good weather has not been enjoyed by all as the north west and Scotland have seen heavy rain .

Article Length Statistics:
count    5000.000000
mean      698.094600
std       343.526824
min        21.000000
25%       447.750000
50%       636.000000
75%       885.000000
max      1955.000000
Name: article_len, dtype: float64

Highlight Length Statistics:
count    5000.000000
mean       51.89960

# SECTION 3. DATA PREPROCESSING AND CLEANING

In [6]:
def clean_text(text):
    text = text.lower()
    # CNN/DailyMail articles often start with "(CNN)" as a dateline tag
    text = re.sub(r'\(cnn\)', '', text)
    # Replace em dashes with space e.g. war--peace -> war peace
    text = re.sub(r'--', ' ', text)
    # Expand contractions
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"n\'t", " not", text)
    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['article_clean'] = df['article'].apply(clean_text)

# Add <sos> and <eos> tokens to highlights
df['highlights_clean'] = df['highlights'].apply(
    lambda x: '<sos> ' + clean_text(x) + ' <eos>'
)

df[['article_clean', 'highlights_clean']].head(3)

,article_clean,highlights_clean
0,by mia de graaf britons flocked to beaches acr...,<sos> people enjoyed temperatures of 17c at br...
1,a couple who weighed a combined 32st were sham...,<sos> couple started piling on pounds after th...
2,video footage shows the heart stopping moment ...,<sos> a 17yearold boy suffering lacerations to...


### TOKENIZATION AND PADDING

In [7]:
MAX_ARTICLE_LEN = 400
MAX_HIGHLIGHT_LEN = 50
VOCAB_SIZE = 20000

# Articles tokenize
art_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
art_tokenizer.fit_on_texts(df['article_clean'])
art_sequences = art_tokenizer.texts_to_sequences(df['article_clean'])
art_padded = pad_sequences(art_sequences, maxlen=MAX_ARTICLE_LEN, padding='post', truncating='post')

# Highlights tokenize
hl_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
hl_tokenizer.fit_on_texts(df['highlights_clean'])
hl_sequences = hl_tokenizer.texts_to_sequences(df['highlights_clean'])
hl_padded = pad_sequences(hl_sequences, maxlen=MAX_HIGHLIGHT_LEN, padding='post', truncating='post')

print("Article shape:", art_padded.shape)
print("Highlight shape:", hl_padded.shape)
print("Article vocab size:", len(art_tokenizer.word_index))
print("Highlight vocab size:", len(hl_tokenizer.word_index))

Article shape: (5000, 400)
Highlight shape: (5000, 50)
Article vocab size: 83502
Highlight vocab size: 26268


### Train/Val/Test Split

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    art_padded, hl_padded, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

Train: (4000, 400)
Val:   (500, 400)
Test:  (500, 400)
